# Track B — Fine-tune Relation Classifier

Phân loại quan hệ **6 lớp** giữa hai atomic claim của hai reviewer:
`AGREEMENT · PARTIAL_AGREEMENT · COMPLEMENTARY · PARTIAL_CONTRADICTION · CONTRADICTION · UNRELATED`

Dữ liệu: `phase2_trackb/processed/trackB_silver.jsonl` (1099 cặp, nhãn đọc tay theo RUBRIC).

---

### Ba lựa chọn thiết kế không phải mặc định

1. **Đối xứng theo cấu trúc.** Quan hệ không phụ thuộc claim nào đứng trước. Toàn bộ sự cố
   order-flip của pipeline ensemble (94% cặp phải đi debate chỉ vì có model đổi ý khi đảo A/B)
   đến từ chỗ này. Xử lý bằng **augment 2 chiều lúc train** + **cộng logit 2 chiều lúc suy luận**,
   và **đo trực tiếp flip-rate** để kiểm chứng chứ không chỉ hi vọng.
2. **Trọng số lớp.** CONTRADICTION chỉ ~2.6%. Không có trọng số thì model bỏ hẳn lớp này
   mà accuracy vẫn đẹp.
3. **Chia theo paper.** Nhiều cặp dùng chung claim/paper — chia ngẫu nhiên sẽ cho metric ảo.

### Thứ tự chạy
`Setup → Data → Baselines → Train → Đánh giá → 5-fold CV → Learning curve → Lưu checkpoint`

## 0 · Setup

In [ ]:
!pip install -q transformers==4.44.2 scikit-learn
import torch, subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

## 1 · Lấy dữ liệu

Repo public nên clone thẳng. **Nhớ push `trackB_silver.jsonl` lên GitHub trước khi chạy cell này.**

In [ ]:
import os, json
REPO = "https://github.com/navihat/build-phase2-finetune.git"
if not os.path.exists("build-phase2-finetune"):
    !git clone -q {REPO}
%cd build-phase2-finetune
!git pull -q
SILVER = "phase2_trackb/processed/trackB_silver.jsonl"

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

rows = read_jsonl(SILVER)
print(f"{len(rows)} cặp | {len({r['paper_id'] for r in rows})} nhóm paper")

## 2 · Cấu hình

In [ ]:
from dataclasses import dataclass

LABELS = ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY",
          "PARTIAL_CONTRADICTION","CONTRADICTION","UNRELATED"]
L2I = {l:i for i,l in enumerate(LABELS)}

# Trục quan hệ: sai giữa hai nhãn KỀ NHAU nhẹ hơn nhiều so với sai giữa hai nhãn XA NHAU.
# UNRELATED nằm ngoài trục, chỉ kề COMPLEMENTARY. (Khớp is_adjacent() của track_b_pipeline)
AXIS = ["AGREEMENT","PARTIAL_AGREEMENT","COMPLEMENTARY","PARTIAL_CONTRADICTION","CONTRADICTION"]

def axis_dist(a, b):
    """Khoảng cách trên trục quan hệ; UNRELATED cách COMPLEMENTARY 1 bước."""
    if a == b: return 0
    if "UNRELATED" in (a, b):
        other = b if a == "UNRELATED" else a
        return 1 if other == "COMPLEMENTARY" else 3
    return abs(AXIS.index(a) - AXIS.index(b))

@dataclass
class Cfg:
    model: str = "roberta-base"    # EN. Có dữ liệu VI -> "xlm-roberta-base"
    epochs: int = 6
    batch: int = 16
    eval_batch: int = 64
    lr: float = 2e-5
    max_len: int = 160
    fp16: bool = True
    symmetric_aug: bool = True     # train: nhân đôi cặp theo 2 thứ tự
    symmetric_tta: bool = True     # infer: cộng logit 2 thứ tự
    seed: int = 20260828

cfg = Cfg()
print(cfg)

## 3 · Chia dữ liệu theo NHÓM PAPER

Cặp UNRELATED chéo paper có `paper_id` dạng `"A|B"` → lấy paper trái làm khoá nhóm.
Không dùng union-find gộp A với B: 110 cạnh chéo trên 263 paper sẽ nối phần lớn paper
thành một thành phần liên thông khổng lồ, làm việc nhóm mất tác dụng.

In [ ]:
import random, collections
import numpy as np

def group_of(r): return r["paper_id"].split("|")[0]

def assign_groups(rows, ratios, seed=cfg.seed):
    by = collections.defaultdict(list)
    for r in rows: by[group_of(r)].append(r)
    groups = sorted(by, key=lambda g: (-len(by[g]), g))
    rnd = random.Random(seed); rnd.shuffle(groups)
    groups.sort(key=lambda g: -len(by[g]))
    n = len(rows); quota = [n*x for x in ratios]
    size = [0]*len(ratios); out = [[] for _ in ratios]
    for g in groups:                       # nhóm lớn trước, luôn bỏ vào phần đang thiếu nhất
        i = min(range(len(ratios)), key=lambda k: (size[k]-quota[k])/max(quota[k],1))
        out[i].extend(by[g]); size[i] += len(by[g])
    return out

train_rows, val_rows, test_rows = assign_groups(rows, [0.8, 0.1, 0.1])

def summarize(name, part):
    d = collections.Counter(r["relation"] for r in part)
    print(f"{name:<7}{len(part):>5}  {len({group_of(r) for r in part}):>4} paper  " +
          "  ".join(f"{l[:4]}:{d.get(l,0)}" for l in LABELS))

for n_, p_ in [("train",train_rows),("val",val_rows),("test",test_rows)]: summarize(n_, p_)

gt,gv,gs = ({group_of(r) for r in p} for p in (train_rows,val_rows,test_rows))
assert not (gt&gv) and not (gt&gs) and not (gv&gs), "paper lọt sang phần khác"
print("\n[OK] không paper nào nằm ở hai phần")
print("[!] test chỉ ~110 cặp, CONTRADICTION rất ít -> đọc số từ 5-fold CV mới đáng tin")

## 4 · Baseline — mốc để so sánh

Không có mốc thì macro-F1 = 0.45 là tốt hay tệ đều không biết.

- **majority**: luôn đoán COMPLEMENTARY (lớp đông nhất)
- **stance-rule**: dùng đúng tín hiệu đã dùng để đào cặp — stance đối nghịch → PARTIAL_CONTRADICTION

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix

y_test = np.array([L2I[r["relation"]] for r in test_rows])

pred_majority = np.full(len(test_rows), L2I["COMPLEMENTARY"])

def stance_rule(r):
    s = {r["left"]["stance"], r["right"]["stance"]}
    if s == {"POSITIVE","NEGATIVE"}: return L2I["PARTIAL_CONTRADICTION"]
    if s == {"POSITIVE"}:            return L2I["PARTIAL_AGREEMENT"]
    return L2I["COMPLEMENTARY"]
pred_stance = np.array([stance_rule(r) for r in test_rows])

for name, p in [("majority", pred_majority), ("stance-rule", pred_stance)]:
    print(f"{name:<12} macro-F1={f1_score(y_test,p,average='macro',zero_division=0):.4f}  "
          f"micro-F1={f1_score(y_test,p,average='micro',zero_division=0):.4f}")

## 5 · Model + vòng huấn luyện

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)

torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
device = torch.device("cuda")

class PairSet(Dataset):
    def __init__(self, rows, tok, max_len, symmetric_aug):
        self.ex = []
        for r in rows:
            y = L2I[r["relation"]]; a, b = r["left"]["text"], r["right"]["text"]
            self.ex.append((a,b,y))
            if symmetric_aug: self.ex.append((b,a,y))   # cùng nhãn, đảo thứ tự
        self.tok, self.max_len = tok, max_len
    def __len__(self): return len(self.ex)
    def __getitem__(self, i):
        a,b,y = self.ex[i]
        e = self.tok(a, b, truncation=True, max_length=self.max_len, padding=False)
        e["labels"] = y; return e

def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b.pop("labels") for b in batch])
        out = tok.pad(batch, return_tensors="pt"); out["labels"] = labels
        return out
    return collate

@torch.no_grad()
def predict_logits(model, tok, rows, symmetric_tta):
    """symmetric_tta=True: cộng logit của cả hai thứ tự (A,B) và (B,A)."""
    model.eval()
    orders = [(0,1)] + ([(1,0)] if symmetric_tta else [])
    total = None
    for lo, ro in orders:
        parts = []
        for i in range(0, len(rows), cfg.eval_batch):
            ch = rows[i:i+cfg.eval_batch]
            t = [(r["left"]["text"], r["right"]["text"]) for r in ch]
            enc = tok([x[lo] for x in t], [x[ro] for x in t], truncation=True,
                      max_length=cfg.max_len, padding=True, return_tensors="pt").to(device)
            parts.append(model(**enc).logits.float().cpu())
        lg = torch.cat(parts)
        total = lg if total is None else total + lg
    return total.numpy()

def train_model(train_rows, cfg, tag=""):
    tok = AutoTokenizer.from_pretrained(cfg.model)
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model, num_labels=len(LABELS)).to(device)
    dl = DataLoader(PairSet(train_rows, tok, cfg.max_len, cfg.symmetric_aug),
                    batch_size=cfg.batch, shuffle=True, collate_fn=make_collate(tok))
    # trọng số lớp inverse-frequency, tính trên chính tập train này
    cnt = collections.Counter(r["relation"] for r in train_rows)
    w = torch.tensor([len(train_rows)/(len(LABELS)*max(cnt.get(l,0),1)) for l in LABELS],
                     dtype=torch.float, device=device)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    steps = len(dl)*cfg.epochs
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(steps*0.1), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)
    for ep in range(cfg.epochs):
        model.train(); run = 0.0
        for batch in dl:
            batch = {k:v.to(device) for k,v in batch.items()}
            labels = batch.pop("labels")
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=cfg.fp16):
                loss = loss_fn(model(**batch).logits.float(), labels)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        print(f"  {tag}epoch {ep+1}/{cfg.epochs}  loss={run/len(dl):.4f}")
    return model, tok

print("sẵn sàng")

## 6 · Train (train+val → test)

In [ ]:
model, tok = train_model(train_rows + val_rows, cfg)
logits_test = predict_logits(model, tok, test_rows, cfg.symmetric_tta)
y_pred = logits_test.argmax(1)
print(f"\nmacro-F1 = {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

## 7 · Đánh giá

Bảy phép đo, mỗi phép trả lời một câu hỏi khác nhau. Đừng chỉ đọc accuracy.

### 7.1 · Per-class P/R/F1 — *lớp nào model bỏ rơi?*

Với phân bố lệch 45% / 2.6%, **accuracy vô dụng**: đoán COMPLEMENTARY hết vẫn được ~45%.
Macro-F1 là số chính, vì nó cho mọi lớp trọng số bằng nhau.

In [ ]:
present = sorted(set(y_test) | set(y_pred))
print(classification_report(y_test, y_pred, labels=present,
      target_names=[LABELS[i] for i in present], digits=3, zero_division=0))
print(f"macro-F1={f1_score(y_test,y_pred,average='macro',zero_division=0):.4f}   "
      f"micro-F1={f1_score(y_test,y_pred,average='micro',zero_division=0):.4f}")

### 7.2 · Ma trận nhầm lẫn — *sai theo MẪU nào?*

Tìm **mẫu lỗi hệ thống**, không phải tỉ lệ %. Một ranh giới lệch đều một hướng
(ví dụ PARTIAL_CONTRADICTION luôn bị đoán thành CONTRADICTION) là dấu hiệu rubric
mờ ở đúng ranh giới đó, không phải model kém.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=list(range(len(LABELS))))
print(" "*26 + "".join(f"{l[:6]:>8}" for l in LABELS) + "     n")
for i,l in enumerate(LABELS):
    print(f"{l:<26}" + "".join(f"{v:>8}" for v in cm[i]) + f"  {cm[i].sum():>5}")

print("\nCác ô nhầm nhiều nhất:")
err = [(cm[i][j], LABELS[i], LABELS[j]) for i in range(6) for j in range(6) if i!=j and cm[i][j]>0]
for n,t,p in sorted(err, reverse=True)[:8]:
    print(f"  {n:>3}x  {t}  ->  {p}   (cách {axis_dist(t,p)} bước trên trục)")

### 7.3 · Độ chính xác có dung sai theo trục — *sai NẶNG hay sai NHẸ?*

6 nhãn nằm trên một trục liên tục. Nhầm `PARTIAL_CONTRADICTION` ↔ `CONTRADICTION`
(kề nhau) nhẹ hơn hẳn nhầm `AGREEMENT` ↔ `CONTRADICTION` (cách 4 bước).
Accuracy thường coi hai lỗi này như nhau — đây là chỗ nó che mất sự thật.

In [ ]:
d = np.array([axis_dist(LABELS[t], LABELS[p]) for t,p in zip(y_test, y_pred)])
print(f"đúng tuyệt đối (d=0)      : {(d==0).mean():.3f}")
print(f"đúng hoặc lệch 1 bước     : {(d<=1).mean():.3f}   <- chỉ số dễ đọc nhất cho ứng dụng")
print(f"lệch >=2 bước (sai nặng)  : {(d>=2).mean():.3f}")
print(f"khoảng cách trung bình    : {d.mean():.3f} bước")
print("\nphân bố khoảng cách lỗi:", dict(collections.Counter(d.tolist())))

### 7.4 · Quadratic Weighted Kappa — *hơn đoán mò bao nhiêu, có tính thứ tự?*

QWK phạt lỗi theo **bình phương khoảng cách** và hiệu chỉnh theo mức đồng thuận ngẫu nhiên.
Đây là thước đo phù hợp nhất cho nhãn có thứ tự. Bỏ UNRELATED vì nó nằm ngoài trục.

Đọc: <0.2 kém · 0.4-0.6 khá · >0.6 tốt · >0.8 rất tốt

In [ ]:
from sklearn.metrics import cohen_kappa_score
ax_idx = {l:i for i,l in enumerate(AXIS)}
mask = np.array([LABELS[t] in ax_idx and LABELS[p] in ax_idx for t,p in zip(y_test,y_pred)])
if mask.sum() > 1:
    yt = [ax_idx[LABELS[t]] for t,m in zip(y_test,mask) if m]
    yp = [ax_idx[LABELS[p]] for p,m in zip(y_pred,mask) if m]
    print(f"QWK trên trục (n={mask.sum()}): {cohen_kappa_score(yt,yp,weights='quadratic'):.4f}")
print(f"Kappa thường (cả 6 lớp)  : {cohen_kappa_score(y_test,y_pred):.4f}")

### 7.5 · Flip-rate — *model có ĐỐI XỨNG thật không?*

**Đây là phép đo quan trọng nhất của dự án này.** Pipeline ensemble sụp đổ vì
Qwen lật nhãn 50% và Gemma 44% khi đảo A/B, khiến 94% cặp phải đi debate.

Đo trên logit THÔ (tắt TTA) để biết model tự nó đã đối xứng chưa. TTA làm flip-rate = 0
theo định nghĩa, nên nếu chỉ đo có TTA thì không phát hiện được vấn đề.

In [ ]:
raw_fwd = predict_logits(model, tok, test_rows, symmetric_tta=False)
swapped = [{**r, "left": r["right"], "right": r["left"]} for r in test_rows]
raw_rev = predict_logits(model, tok, swapped, symmetric_tta=False)
f_, r_ = raw_fwd.argmax(1), raw_rev.argmax(1)
flip = (f_ != r_)
print(f"flip-rate (thô, không TTA) : {flip.mean():.3f}   ({flip.sum()}/{len(flip)} cặp)")
if flip.sum():
    dd = [axis_dist(LABELS[a], LABELS[b]) for a,b in zip(f_[flip], r_[flip])]
    print(f"  trong đó lệch 1 bước     : {sum(1 for x in dd if x==1)}/{len(dd)}")
print("\nĐối chiếu pipeline ensemble cũ: Qwen 50%, Gemma 44%, SeaLLM 18%.")
print("Dưới ~10% là đã khắc phục được vấn đề đã làm hỏng Track B.")

### 7.6 · Tách theo NGUỒN dữ liệu — *con số nào là thật?*

Dữ liệu đến từ 4 nguồn có độ tin cậy khác nhau. Nếu model đạt 99% trên
`synthetic_cross_paper` (UNRELATED sinh bằng luật, rất dễ) nhưng kém ở nơi khác thì
**con số UNRELATED trong báo cáo tổng là ảo**. Bảng này bắt đúng loại tự lừa đó.

In [ ]:
by_src = collections.defaultdict(list)
for i,r in enumerate(test_rows): by_src[r["source"]].append(i)
print(f"{'nguồn':<28}{'n':>5}{'acc':>8}{'macroF1':>9}")
for s, idx in sorted(by_src.items()):
    yt_, yp_ = y_test[idx], y_pred[idx]
    print(f"{s:<28}{len(idx):>5}{(yt_==yp_).mean():>8.3f}"
          f"{f1_score(yt_,yp_,average='macro',zero_division=0):>9.3f}")

### 7.7 · Hiệu chuẩn — *có thể đặt ngưỡng ABSTAIN không?*

Nếu model tự tin sai nhiều thì không dùng được confidence để lọc. Bảng này cho biết
nên cắt ngưỡng ở đâu nếu muốn đánh đổi coverage lấy độ chính xác.

In [ ]:
probs = torch.softmax(torch.tensor(logits_test), dim=1).numpy()
conf = probs.max(1); correct = (y_pred == y_test)
print(f"confidence trung bình: đúng={conf[correct].mean():.3f}  sai={conf[~correct].mean():.3f}")
print(f"\n{'ngưỡng':>8}{'coverage':>11}{'acc trên phần giữ lại':>24}")
for t in [0.0,0.5,0.6,0.7,0.8,0.9]:
    m = conf >= t
    if m.sum():
        print(f"{t:>8.1f}{m.mean():>11.3f}{correct[m].mean():>24.3f}")

# ECE 10 bin
bins = np.linspace(0,1,11); ece = 0.0
for lo,hi in zip(bins[:-1],bins[1:]):
    m = (conf>lo)&(conf<=hi)
    if m.sum(): ece += m.mean()*abs(correct[m].mean()-conf[m].mean())
print(f"\nECE = {ece:.4f}   (<0.05 hiệu chuẩn tốt, >0.15 quá tự tin)")

## 8 · 5-fold CV — con số đáng tin

Test chỉ 110 cặp, CONTRADICTION đúng 2 mẫu → F1 lớp đó trên test là số ngẫu nhiên.
CV cho **mỗi cặp trong 1099 cặp được dự đoán đúng một lần**, nên mọi lớp đều đủ mẫu.

**Đây mới là con số để báo cáo.** Chạy ~5× lâu hơn.

In [ ]:
N_FOLDS = 5
parts = assign_groups(rows, [1/N_FOLDS]*N_FOLDS)
fold_of = {r["pair_id"]: k for k,p in enumerate(parts) for r in p}

all_t, all_p, fold_f1 = [], [], []
for k in range(N_FOLDS):
    tr = [r for r in rows if fold_of[r["pair_id"]] != k]
    te = [r for r in rows if fold_of[r["pair_id"]] == k]
    print(f"\n--- fold {k}: train={len(tr)} test={len(te)} ---")
    m_, t_ = train_model(tr, cfg, tag=f"[f{k}] ")
    yp_ = predict_logits(m_, t_, te, cfg.symmetric_tta).argmax(1)
    yt_ = np.array([L2I[r["relation"]] for r in te])
    s = f1_score(yt_, yp_, average="macro", zero_division=0); fold_f1.append(s)
    print(f"  fold {k} macro-F1 = {s:.4f}")
    all_t.append(yt_); all_p.append(yp_)
    del m_; torch.cuda.empty_cache()

cv_t, cv_p = np.concatenate(all_t), np.concatenate(all_p)
print("\n" + "="*62)
print(f"macro-F1 từng fold: {[f'{s:.3f}' for s in fold_f1]}")
print(f"trung bình = {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f}")
print("="*62)
print(classification_report(cv_t, cv_p, target_names=LABELS, digits=3, zero_division=0))
dcv = np.array([axis_dist(LABELS[t],LABELS[p]) for t,p in zip(cv_t,cv_p)])
print(f"đúng hoặc lệch 1 bước: {(dcv<=1).mean():.3f}   sai nặng (>=2): {(dcv>=2).mean():.3f}")

## 9 · Learning curve — *thêm dữ liệu có đáng không?*

Trả lời bằng số cho câu "1099 cặp đã đủ chưa": train trên 25/50/75/100% rồi nhìn độ dốc.
Còn dốc → gán thêm nhãn sẽ có lời. Đã phẳng → tiền nên đổ vào chỗ khác (model lớn hơn,
sửa rubric, cân bằng lớp).

In [ ]:
curve = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    sub_groups = assign_groups(train_rows + val_rows, [frac, 1-frac])[0] if frac < 1 \
                 else train_rows + val_rows
    m_, t_ = train_model(sub_groups, cfg, tag=f"[{int(frac*100)}%] ")
    yp_ = predict_logits(m_, t_, test_rows, cfg.symmetric_tta).argmax(1)
    s = f1_score(y_test, yp_, average="macro", zero_division=0)
    curve.append((len(sub_groups), s)); print(f"  n={len(sub_groups):<5} macro-F1={s:.4f}")
    del m_; torch.cuda.empty_cache()

print("\n n_train   macro-F1   Δ so với mức trước")
prev = None
for n_,s_ in curve:
    print(f"{n_:>8}{s_:>11.4f}" + (f"{s_-prev:>+12.4f}" if prev is not None else ""))
    prev = s_
print("\nΔ cuối còn lớn -> gán thêm nhãn còn lời. Δ ~0 -> đã bão hoà.")

## 10 · Lưu checkpoint về Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/phase2_trackb/relation_classifier_v1"
model.save_pretrained(OUT); tok.save_pretrained(OUT)
with open(OUT + "/labels.json", "w") as f:
    json.dump({"labels": LABELS, "symmetric_tta": cfg.symmetric_tta}, f, indent=1)
print("đã lưu ->", OUT)
print("\nInference tại máy (GTX 1650 4GB thừa sức):")
print("  m = AutoModelForSequenceClassification.from_pretrained(OUT)")
print("  # nhớ cộng logit cả hai thứ tự (A,B) và (B,A) như lúc đánh giá")